# 03 — Baseline Training

**Goal:** Train a minimal semantic segmentation CNN from scratch to verify the full pipeline runs end-to-end.

---

## What model are we using?

This notebook uses a **very simple 3-layer CNN** as a placeholder.  
Its purpose is to confirm that:
- Data loading works correctly.
- Tensor shapes are correct at every step.
- The loss goes down (even slightly) after one epoch.
- Checkpointing saves and restores weights.

This model will **not** produce good segmentations — that is expected.  
Once the pipeline is verified, replace the model with U-Net, FCN, or DeepLabv3.

> **Prerequisite:** Run notebooks 00–02 first and ensure `pairs` is populated.

In [ ]:
import hashlib
import os
import platform
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data_paths import OUTPUTS_DIR, MODELS_DIR, LOGS_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, load_pairs_from_manifest
from src.train_utils import get_device, save_checkpoint, train_one_epoch, evaluate

ensure_project_dirs()

print(f"PyTorch version: {torch.__version__}")

All project directories are ready.
PyTorch version: 2.12.1+cu126


## Step 1 — Load Pairs

In [ ]:
APPROVED_MANIFEST = OUTPUTS_DIR / "predictions" / "audit" / "approved_nav_pairs.csv"

pairs = load_pairs_from_manifest(
    APPROVED_MANIFEST,
    required_label_scheme="NAV",
    require_shape_match=True,
)

print(f"Approved NAV pairs loaded: {len(pairs)}")
print(f"Manifest path: {APPROVED_MANIFEST}")

if len(pairs) == 0:
    raise RuntimeError(
        "Approved NAV manifest contains no usable pairs. "
        "Build and validate approved_nav_pairs.csv from notebook 02 first."
    )


Total pairs (all schemes): 46460
Total pairs (NAV only)   : 37958


## Step 2 — Hyperparameters

In [ ]:
IMAGE_SIZE   = (256, 256)   # (width, height) - PIL convention
BATCH_SIZE   = 4
NUM_CLASSES  = 4            # soil, bedrock, sand, big_rock
IGNORE_INDEX = 255          # unlabeled pixels in AI4Mars masks
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 3            # quick quality check for weighted + pretrained baseline

## Step 3 — Create Train/Val Split and DataLoaders

In [ ]:
# Immutable, metadata-aware split from approved manifest pairs.
# All labels derived from the same source-image stem stay in one partition.
from src.dataset import split_pairs_by_source_stem

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SPLIT_SEED = 42

splits = split_pairs_by_source_stem(
    pairs,
    train_ratio=TRAIN_SPLIT,
    val_ratio=VAL_SPLIT,
    test_ratio=TEST_SPLIT,
    seed=SPLIT_SEED,
)

train_pairs = splits["train"]
val_pairs = splits["val"]
test_pairs = splits["test"]

# Create the datasets from the partitioned pair lists with strict geometry checks.
train_dataset = AI4MarsDataset(train_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
val_dataset = AI4MarsDataset(val_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
test_dataset = AI4MarsDataset(test_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")

# DataLoader performance knobs:
cpu_count = os.cpu_count() or 2
if torch.cuda.is_available():
    NUM_WORKERS = max(2, min(8, cpu_count - 1))
else:
    NUM_WORKERS = min(4, max(1, cpu_count // 2))

PIN_MEMORY = torch.cuda.is_available()

print(f"DataLoader num_workers: {NUM_WORKERS}")
print(f"DataLoader pin_memory : {PIN_MEMORY}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

print("Split strategy: deterministic source-stem grouping from approved manifest")
print(f"Split seed   : {SPLIT_SEED}")
print(f"Split ratios : train={TRAIN_SPLIT:.2f}, val={VAL_SPLIT:.2f}, test={TEST_SPLIT:.2f}")

Train samples: 34163
Val samples  : 3795
DataLoader num_workers: 7
DataLoader pin_memory : True
Split strategy: random_split on NAV-only pairs with a fixed manual seed (42)


## Step 3.5 — Compute Data-Driven Class Weights

Estimate class frequencies from the training split so the loss reflects the
actual NAV label distribution instead of an approximate hand-tuned vector.

In [6]:
stats_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

class_pixel_counts = torch.zeros(NUM_CLASSES, dtype=torch.long)
for _, masks in stats_loader:
    valid = masks != IGNORE_INDEX
    for class_id in range(NUM_CLASSES):
        class_pixel_counts[class_id] += ((masks == class_id) & valid).sum()

total_labeled_pixels = class_pixel_counts.sum().item()
class_frequencies = class_pixel_counts.float() / max(total_labeled_pixels, 1)
class_weights = 1.0 / torch.clamp(class_frequencies, min=1e-8)
class_weights = class_weights / class_weights.mean()

print("Train-split labeled pixel counts by class:")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(
        f"  class {class_id} ({class_name:>10s}): {class_pixel_counts[class_id].item():>12d} pixels "
        f"({class_frequencies[class_id].item() * 100:6.2f}%)"
    )

print("\nEmpirical class weights (inverse-frequency, mean-normalized):")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(f"  class {class_id} ({class_name:>10s}): {class_weights[class_id].item():.4f}")

Train-split labeled pixel counts by class:
  class 0 (      soil):    574428237 pixels ( 39.61%)
  class 1 (   bedrock):    415157397 pixels ( 28.63%)
  class 2 (      sand):    398192615 pixels ( 27.46%)
  class 3 (  big_rock):     62455789 pixels (  4.31%)

Empirical class weights (inverse-frequency, mean-normalized):
  class 0 (      soil): 0.3071
  class 1 (   bedrock): 0.4250
  class 2 (      sand): 0.4431
  class 3 (  big_rock): 2.8248


## Step 4 — Verify Batch Shapes

Always check tensor shapes **before** training.  
A shape mismatch will cause a confusing error inside the model.

In [7]:
images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, H, W])")
print(f"Mask  batch shape : {masks.shape}    (expected [B, H, W])")
print(f"Image dtype       : {images.dtype}   (expected float32)")
print(f"Mask  dtype       : {masks.dtype}    (expected int64 / long)")
print(f"Image value range : [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])")

Image batch shape : torch.Size([4, 3, 256, 256])   (expected [B, 3, H, W])
Mask  batch shape : torch.Size([4, 256, 256])    (expected [B, H, W])
Image dtype       : torch.float32   (expected float32)
Mask  dtype       : torch.int64    (expected int64 / long)
Image value range : [0.012, 1.000]  (expected [0, 1])


## Step 5 — Define a Stronger Pretrained Segmentation Model

For the first real baseline, use a pretrained U-Net encoder so the model starts with useful low-level visual features (edges, textures) instead of learning everything from scratch.

In [8]:
import segmentation_models_pytorch as smp

# Use a lighter encoder on CPU to keep iterations practical.
device = get_device()
encoder_name = "mobilenet_v2" if device.type == "cpu" else "resnet34"

model = smp.Unet(
    encoder_name=encoder_name,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

# Quick shape check before training
dummy = torch.randn(2, 3, IMAGE_SIZE[1], IMAGE_SIZE[0]).to(device)
out = model(dummy)
print(f"Encoder: {encoder_name}")
print(f"Model output shape: {out.shape}   (expected [2, {NUM_CLASSES}, {IMAGE_SIZE[1]}, {IMAGE_SIZE[0]}])")

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

Using device: cuda
Encoder: resnet34
Model output shape: torch.Size([2, 4, 256, 256])   (expected [2, 4, 256, 256])
Model parameters: 24,436,804


## Step 6 — Loss Function and Optimiser

In [9]:
# Weighted cross-entropy to counter class imbalance using the empirical class
# distribution computed above.
weights = class_weights.to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=IGNORE_INDEX)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Class weights: {weights.tolist()}")

Class weights: [0.30713504552841187, 0.42496421933174133, 0.44306960701942444, 2.824831008911133]


## Reproducibility Log

Record the key settings needed to reproduce the baseline experiment and the
saved checkpoint.

In [10]:
reproducibility_log = {
    "hardware": {
        "device": str(device),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    },
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "model": {
        "type": "U-Net",
        "encoder": encoder_name,
        "encoder_weights": "imagenet",
        "num_classes": NUM_CLASSES,
    },
    "optimizer": {
        "name": "Adam",
        "learning_rate": LEARNING_RATE,
    },
    "epochs": NUM_EPOCHS,
    "seed": 42,
    "split_strategy": "random_split on NAV-only pairs with manual_seed(42) and VAL_SPLIT=0.1",
    "dataset_version": "ai4mars-dataset-merged-0.6 (Zenodo 15995036)",
    "checkpoint_name": "weighted_unet_epoch03.pth",
}

reproducibility_path = LOGS_DIR / "reproducibility_log.json"
with open(reproducibility_path, "w", encoding="utf-8") as f:
    json.dump(reproducibility_log, f, indent=2)

print(f"Saved reproducibility log to {reproducibility_path}")
print(json.dumps(reproducibility_log, indent=2))

Saved reproducibility log to C:\Users\Jacob\AI4Mars\outputs\logs\reproducibility_log.json
{
  "hardware": {
    "device": "cuda",
    "platform": "Windows-11-10.0.26200-SP0",
    "python": "3.14.3",
    "torch": "2.12.1+cu126",
    "cuda_available": true,
    "cuda_device": "NVIDIA GeForce GTX 1080 Ti"
  },
  "image_size": [
    256,
    256
  ],
  "batch_size": 4,
  "model": {
    "type": "U-Net",
    "encoder": "resnet34",
    "encoder_weights": "imagenet",
    "num_classes": 4
  },
  "optimizer": {
    "name": "Adam",
    "learning_rate": 0.001
  },
  "epochs": 3,
  "seed": 42,
  "split_strategy": "random_split on NAV-only pairs with manual_seed(42) and VAL_SPLIT=0.1",
  "dataset_version": "ai4mars-dataset-merged-0.6 (Zenodo 15995036)",
  "checkpoint_name": "weighted_unet_epoch03.pth"
}


## Step 7 — Training Loop

In [11]:
CLASS_NAMES = ["soil", "bedrock", "sand", "big_rock"]

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics = evaluate(
        model,
        val_loader,
        loss_fn,
        device,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
        return_per_class_iou=(epoch == NUM_EPOCHS),  # avoid extra overhead each epoch
    )

    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val loss   : {val_metrics['val_loss']:.4f}")
    print(f"  Pixel acc  : {val_metrics['pixel_acc']:.4f}")
    print(f"  Mean IoU   : {val_metrics['mean_iou']:.4f}")

    per_class_iou = val_metrics.get("per_class_iou")
    if per_class_iou is not None:
        print("  Per-class IoU:")
        valid_scores = []
        for class_name, score in zip(CLASS_NAMES, per_class_iou):
            if score is None:
                print(f"    - {class_name:8s}: n/a")
            else:
                print(f"    - {class_name:8s}: {score:.4f}")
                valid_scores.append((class_name, score))

        if valid_scores:
            hardest_class, hardest_score = min(valid_scores, key=lambda x: x[1])
            print(f"  Hardest class this epoch: {hardest_class} (IoU={hardest_score:.4f})")


Epoch 1/3
----------------------------------------
  Batch 10/8541  loss=1.4718
  Batch 20/8541  loss=1.3374
  Batch 30/8541  loss=1.3550
  Batch 40/8541  loss=1.2287
  Batch 50/8541  loss=1.2734
  Batch 60/8541  loss=1.0348
  Batch 70/8541  loss=1.7449
  Batch 80/8541  loss=1.1818
  Batch 90/8541  loss=1.9796
  Batch 100/8541  loss=1.4063
  Batch 110/8541  loss=1.1172
  Batch 120/8541  loss=1.4316
  Batch 130/8541  loss=1.3936
  Batch 140/8541  loss=1.1348
  Batch 150/8541  loss=1.1283
  Batch 160/8541  loss=1.1915
  Batch 170/8541  loss=1.2990
  Batch 180/8541  loss=1.0722
  Batch 190/8541  loss=1.3676
  Batch 200/8541  loss=1.0377
  Batch 210/8541  loss=1.2404
  Batch 220/8541  loss=1.2335
  Batch 230/8541  loss=1.1157
  Batch 240/8541  loss=1.4006
  Batch 250/8541  loss=1.2681
  Batch 260/8541  loss=1.1866
  Batch 270/8541  loss=1.0093
  Batch 280/8541  loss=0.9257
  Batch 290/8541  loss=1.2178
  Batch 300/8541  loss=1.1208
  Batch 310/8541  loss=1.2228
  Batch 320/8541  loss=1.05

## Step 8 — Save Checkpoint

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_fingerprint_payload = "\n".join(
    f"{img}|{msk}" for img, msk in val_pairs
)

checkpoint_metadata = {
    "split_strategy": "approved_manifest_grouped_source_stem",
    "seed": SPLIT_SEED,
    "train_ratio": TRAIN_SPLIT,
    "val_ratio": VAL_SPLIT,
    "test_ratio": TEST_SPLIT,
    "nav_only": True,
    "val_pair_count": len(val_pairs),
    "val_split_fingerprint_sha256": hashlib.sha256(split_fingerprint_payload.encode("utf-8")).hexdigest(),
    "manifest_path": str(APPROVED_MANIFEST),
    "manifest_sha256": sha256_file(APPROVED_MANIFEST),
    "approved_pair_count": len(pairs),
}

checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"
save_checkpoint(
    model,
    optimizer,
    epoch=NUM_EPOCHS,
    path=checkpoint_path,
    metadata=checkpoint_metadata,
)
print(f"Saved checkpoint to {checkpoint_path}")
print("Checkpoint metadata fingerprint embedded for strict evaluation matching.")

Checkpoint saved → C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth
Saved checkpoint to C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth


## Next Steps

- Open `04_evaluation_error_analysis.ipynb` to load the checkpoint and analyse predictions.
- Run a cleaner experiment series next:
    - **Baseline:** U-Net with a ResNet34 encoder.
    - **Alternative encoder:** U-Net with EfficientNet if it fits local hardware.
    - **Alternative decoder family:** DeepLabV3+ for stronger context aggregation.
    - **Loss experiments:** Dice, Focal, or CE+Dice hybrids.